# ICU Mortality Prediction — Exploratory Data Analysis

This **self-contained** notebook performs a complete exploratory data analysis for
in-hospital mortality prediction using OMOP CDM data.

1. **OMOP concept exploration** — browse available domains, vocabularies, measurements
2. **Cohort extraction** — select eligible visits via SQL
3. **Feature engineering** — extract H0–H24 measurements, aggregate, pivot to wide format
4. **Cohort overview** — demographics, mortality rate, admission timeline
5. **Feature distributions** — vitals, labs, neuro by outcome
6. **Missing data analysis** — patterns and clinical impact
7. **Correlation analysis** — feature relationships and multicollinearity
8. **Outcome analysis** — Table 1, box plots, univariate associations
9. **Outlier detection** — clinical plausibility checks

**Requirements:** An active database connection with OMOP CDM tables
(person, visit_occurrence, death, measurement, concept).

> `sql_query(sql)` is automatically available in Linkr notebooks.
> It queries the active DuckDB connection and returns a pandas DataFrame.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

---
## 1. OMOP Concept Exploration

Before building any model, we need to understand what data is available
in the OMOP CDM. The `concept` table contains all standardized medical concepts.

In [ ]:
# How many concepts per domain?
domain_counts = await sql_query("""
    SELECT domain_id, COUNT(*) AS n_concepts
    FROM concept
    WHERE invalid_reason IS NULL
    GROUP BY domain_id
    ORDER BY n_concepts DESC
""")
domain_counts

In [ ]:
# Top vocabularies
vocab_counts = await sql_query("""
    SELECT vocabulary_id, COUNT(*) AS n_concepts
    FROM concept
    WHERE invalid_reason IS NULL
    GROUP BY vocabulary_id
    ORDER BY n_concepts DESC
    LIMIT 20
""")
vocab_counts

In [ ]:
# Measurement concepts that actually have data (>= 100 records)
measurement_concepts = await sql_query("""
    SELECT
        c.concept_id, c.concept_name, c.vocabulary_id,
        COUNT(*) AS n_records,
        COUNT(DISTINCT m.person_id) AS n_patients,
        ROUND(AVG(m.value_as_number), 2) AS mean_value,
        ROUND(STDDEV(m.value_as_number), 2) AS std_value
    FROM measurement m
    JOIN concept c ON m.measurement_concept_id = c.concept_id
    WHERE m.value_as_number IS NOT NULL
    GROUP BY c.concept_id, c.concept_name, c.vocabulary_id
    HAVING COUNT(*) >= 100
    ORDER BY n_records DESC
    LIMIT 40
""")
print(f"Measurement concepts with >= 100 records: {len(measurement_concepts)}")
measurement_concepts

In [ ]:
# Concepts selected for this study
VITALS = {
    3027018: "hr",           # Heart rate
    3004249: "sbp",          # Systolic blood pressure
    3012888: "dbp",          # Diastolic blood pressure
    3027598: "mbp",          # Mean blood pressure
    3024171: "resp_rate",    # Respiratory rate
    40762499: "spo2",        # SpO2
    3020891: "temp",         # Temperature
}
LABS = {
    3000963: "hemoglobin",   3023314: "hematocrit",  3024929: "platelets",
    3003282: "wbc",          3019550: "sodium",      3023103: "potassium",
    3014576: "chloride",     3016293: "bicarbonate", 3016723: "creatinine",
    3013682: "bun",          3004501: "glucose",     3037278: "anion_gap",
    3015377: "calcium",      3012095: "magnesium",   3011904: "phosphate",
}
NEURO = {3016335: "gcs_eye", 3009094: "gcs_verbal", 3008223: "gcs_motor"}
ALL_MEASUREMENTS = {**VITALS, **LABS, **NEURO}

concept_ids = ", ".join(str(cid) for cid in ALL_MEASUREMENTS.keys())
selected_info = await sql_query(f"""
    SELECT c.concept_id, c.concept_name, c.vocabulary_id, c.domain_id,
           COUNT(*) AS n_records, COUNT(DISTINCT m.person_id) AS n_patients
    FROM measurement m
    JOIN concept c ON m.measurement_concept_id = c.concept_id
    WHERE c.concept_id IN ({concept_ids}) AND m.value_as_number IS NOT NULL
    GROUP BY c.concept_id, c.concept_name, c.vocabulary_id, c.domain_id
    ORDER BY n_records DESC
""")
print(f"Selected concepts found in data: {len(selected_info)} / {len(ALL_MEASUREMENTS)}")
selected_info

---
## 2. Cohort Extraction

We define our study cohort using SQL views:
- Hospital stays >= 24 hours
- At least one numeric measurement in the first 24 hours
- In-hospital mortality = death between admission and discharge + 1 day

In [ ]:
# Eligible visits: stays >= 24 hours
await sql_query("""
    CREATE OR REPLACE VIEW eligible_visits AS
    SELECT
        v.visit_occurrence_id, v.person_id, v.visit_concept_id,
        v.visit_start_date,
        v.visit_start_datetime::TIMESTAMP AS visit_start_datetime,
        v.visit_end_date,
        v.visit_end_datetime::TIMESTAMP AS visit_end_datetime,
        v.discharge_to_concept_id,
        EXTRACT(EPOCH FROM (v.visit_end_datetime::TIMESTAMP
            - v.visit_start_datetime::TIMESTAMP)) / 3600 AS los_hours
    FROM visit_occurrence v
    WHERE EXTRACT(EPOCH FROM (v.visit_end_datetime::TIMESTAMP
        - v.visit_start_datetime::TIMESTAMP)) / 3600 >= 24
""")

# In-hospital mortality flag
await sql_query("""
    CREATE OR REPLACE VIEW visit_mortality AS
    SELECT ev.*,
        CASE WHEN d.death_date IS NOT NULL
             AND d.death_date BETWEEN ev.visit_start_date
                                   AND ev.visit_end_date + INTERVAL '1 day'
             THEN 1 ELSE 0 END AS in_hospital_death
    FROM eligible_visits ev
    LEFT JOIN death d ON ev.person_id = d.person_id
""")

# Final cohort: demographics + require >= 1 measurement in H0-H24
await sql_query("""
    CREATE OR REPLACE VIEW cohort AS
    SELECT
        vm.visit_occurrence_id, vm.person_id, vm.visit_start_datetime,
        EXTRACT(YEAR FROM vm.visit_start_date) - p.year_of_birth AS age,
        p.gender_source_value AS sex,
        vm.los_hours, vm.in_hospital_death
    FROM visit_mortality vm
    JOIN person p ON vm.person_id = p.person_id
    WHERE EXISTS (
        SELECT 1 FROM measurement m
        WHERE m.visit_occurrence_id = vm.visit_occurrence_id
          AND m.value_as_number IS NOT NULL
          AND m.measurement_datetime::TIMESTAMP >= vm.visit_start_datetime
          AND m.measurement_datetime::TIMESTAMP
              <= vm.visit_start_datetime + INTERVAL '24 hours'
    )
""")

# Quick check
summary = await sql_query("""
    SELECT COUNT(*) AS n_visits, COUNT(DISTINCT person_id) AS n_patients,
        SUM(in_hospital_death) AS n_deaths,
        ROUND(100.0 * SUM(in_hospital_death) / COUNT(*), 1) AS mortality_pct,
        ROUND(AVG(age), 1) AS mean_age,
        ROUND(AVG(los_hours), 1) AS mean_los_hours
    FROM cohort
""")
print("Cohort summary:")
summary

---
## 3. Feature Engineering

Extract measurements from H0-H24, aggregate per visit, and pivot from OMOP
long format to one-row-per-visit wide format.

| Category | Aggregation |
|---|---|
| Vitals (7) | mean, min, max |
| Labs (15) | first value |
| Neuro / GCS (3) | minimum (worst) |

In [ ]:
# Extract all measurements in H0-H24 for cohort visits
concept_ids = ", ".join(str(cid) for cid in ALL_MEASUREMENTS.keys())
measurements_h24 = await sql_query(f"""
    SELECT m.visit_occurrence_id, m.measurement_concept_id,
           m.value_as_number,
           m.measurement_datetime::TIMESTAMP AS measurement_datetime,
           c.visit_start_datetime
    FROM measurement m
    JOIN cohort c ON m.visit_occurrence_id = c.visit_occurrence_id
    WHERE m.measurement_concept_id IN ({concept_ids})
      AND m.value_as_number IS NOT NULL
      AND m.measurement_datetime::TIMESTAMP >= c.visit_start_datetime
      AND m.measurement_datetime::TIMESTAMP
          <= c.visit_start_datetime + INTERVAL '24 hours'
""")
print(f"Measurements in H0-H24: {len(measurements_h24)} rows")

In [ ]:
# Aggregate per visit x measurement
measurements_h24["feature"] = measurements_h24["measurement_concept_id"].map(ALL_MEASUREMENTS)
vitals_ids, labs_ids, neuro_ids = set(VITALS), set(LABS), set(NEURO)

aggregated_rows = []
for (visit_id, feature), group in measurements_h24.groupby(
    ["visit_occurrence_id", "feature"]
):
    cid = group["measurement_concept_id"].iloc[0]
    values = group.sort_values("measurement_datetime")["value_as_number"]
    if cid in vitals_ids:
        for agg, fn in [("mean", values.mean), ("min", values.min), ("max", values.max)]:
            aggregated_rows.append({"visit_occurrence_id": visit_id,
                "col": f"{feature}_{agg}", "val": fn()})
    elif cid in neuro_ids:
        aggregated_rows.append({"visit_occurrence_id": visit_id,
            "col": f"{feature}_min", "val": values.min()})
    elif cid in labs_ids:
        aggregated_rows.append({"visit_occurrence_id": visit_id,
            "col": f"{feature}_first", "val": values.iloc[0]})

agg_df = pd.DataFrame(aggregated_rows)
print(f"Aggregated features: {len(agg_df)} rows, "
      f"{agg_df['col'].nunique()} distinct columns")

In [ ]:
# Pivot to wide format and merge with cohort demographics
wide = agg_df.pivot_table(
    index="visit_occurrence_id", columns="col",
    values="val", aggfunc="first"
).reset_index()

cohort_df = await sql_query("""
    SELECT visit_occurrence_id, person_id, age, sex, los_hours,
           in_hospital_death, visit_start_datetime
    FROM cohort
""")

df = cohort_df.merge(wide, on="visit_occurrence_id", how="left")

# Sort columns: identifiers, demographics, outcome, features
id_cols = ["visit_occurrence_id", "person_id"]
demo_cols = ["age", "sex", "los_hours"]
outcome_cols = ["in_hospital_death"]
meta_cols = ["visit_start_datetime"]
feature_cols = sorted([c for c in df.columns
    if c not in id_cols + demo_cols + outcome_cols + meta_cols])
df = df[id_cols + demo_cols + outcome_cols + meta_cols + feature_cols]

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Deaths: {int(df['in_hospital_death'].sum())} / {len(df)} "
      f"({100 * df['in_hospital_death'].mean():.1f}%)")

In [ ]:
# Export CSV for the ML notebook (02_ml_mortality.qmd)
export_cols = [c for c in df.columns if c != "visit_start_datetime"]
df[export_cols].to_csv("data/datasets/mortality_dataset.csv", index=False)
print("Dataset saved to data/datasets/mortality_dataset.csv")

---
## 4. Cohort Overview

In [ ]:
print(f"Cohort: {len(df)} visits, {df['person_id'].nunique()} unique patients")
print(f"Mortality: {int(df['in_hospital_death'].sum())} deaths "
      f"({100 * df['in_hospital_death'].mean():.1f}%)")
print(f"\nAge: {df['age'].mean():.1f} +/- {df['age'].std():.1f} years "
      f"(range {df['age'].min():.0f}-{df['age'].max():.0f})")
print(f"LOS: {df['los_hours'].mean():.0f} +/- {df['los_hours'].std():.0f} hours "
      f"(median {df['los_hours'].median():.0f})")
print(f"\nSex distribution:")
print(df['sex'].value_counts().to_string())

In [ ]:
# Age distribution, LOS, and mortality by age group
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for sex, color in [('M', '#4C78A8'), ('F', '#E45756')]:
    subset = df[df['sex'] == sex]
    axes[0].hist(subset['age'], bins=20, alpha=0.6, label=sex,
                 color=color, edgecolor='white')
axes[0].set_xlabel('Age (years)'); axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution by Sex'); axes[0].legend()

axes[1].hist(df['los_hours'], bins=50, color='#72B7B2', edgecolor='white')
axes[1].set_xlabel('Length of Stay (hours)'); axes[1].set_ylabel('Count')
axes[1].set_title('Length of Stay Distribution'); axes[1].set_yscale('log')

df['age_group'] = pd.cut(df['age'], bins=[0, 40, 50, 60, 70, 80, 120],
                          labels=['<40', '40-49', '50-59', '60-69', '70-79', '>=80'])
mort_by_age = df.groupby('age_group', observed=True)['in_hospital_death'].mean() * 100
mort_by_age.plot(kind='bar', ax=axes[2], color='#F58518', edgecolor='white')
axes[2].set_xlabel('Age Group'); axes[2].set_ylabel('Mortality (%)')
axes[2].set_title('Mortality Rate by Age Group')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Mortality by sex
mort_by_sex = df.groupby('sex')['in_hospital_death'].agg(['sum', 'count', 'mean'])
mort_by_sex.columns = ['Deaths', 'Total', 'Mortality Rate']
mort_by_sex['Mortality Rate'] = (mort_by_sex['Mortality Rate'] * 100).round(1)
mort_by_sex

In [ ]:
# Admission timeline
df['admit_date'] = pd.to_datetime(df['visit_start_datetime']).dt.date
daily = df.groupby('admit_date').agg(
    admissions=('person_id', 'count'),
    deaths=('in_hospital_death', 'sum')
).reset_index()
daily['admit_date'] = pd.to_datetime(daily['admit_date'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily['admit_date'], daily['admissions'],
        color='#4C78A8', lw=0.8, label='Admissions')
ax.fill_between(daily['admit_date'], daily['deaths'],
                color='#E45756', alpha=0.5, label='Deaths')
ax.set_xlabel('Date'); ax.set_ylabel('Count')
ax.set_title('Admissions and Deaths Over Time'); ax.legend()
plt.tight_layout(); plt.show()

---
## 5. Feature Distributions

In [ ]:
df.describe().round(2)

In [ ]:
# Vital signs distribution by outcome
alive = df[df['in_hospital_death'] == 0]
dead = df[df['in_hospital_death'] == 1]

vitals_cols = [c for c in df.columns
    if any(c.startswith(v) for v in ['hr_', 'sbp_', 'dbp_',
        'resp_rate_', 'spo2_', 'temp_'])
    and c.endswith('_mean')]

n = len(vitals_cols)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(vitals_cols):
    axes[i].hist(alive[col].dropna(), bins=30, alpha=0.6, color='#4C78A8',
                 label='Alive', density=True, edgecolor='white')
    axes[i].hist(dead[col].dropna(), bins=30, alpha=0.6, color='#E45756',
                 label='Dead', density=True, edgecolor='white')
    axes[i].set_title(col.replace('_mean', '').upper())
    axes[i].set_ylabel('Density')
    if i == 0: axes[i].legend()
for j in range(i + 1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Vital Signs (H0-H24 Mean) by Outcome', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Lab values distribution by outcome
labs_cols = [c for c in df.columns if c.endswith('_first') and
    any(c.startswith(v) for v in ['hemoglobin', 'creatinine', 'potassium',
        'sodium', 'glucose', 'bun', 'wbc', 'platelets', 'bicarbonate',
        'anion_gap', 'calcium', 'hematocrit', 'chloride',
        'magnesium', 'phosphate'])]

n = len(labs_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
axes = axes.flatten()
for i, col in enumerate(labs_cols):
    axes[i].hist(alive[col].dropna(), bins=30, alpha=0.6, color='#4C78A8',
                 label='Alive', density=True, edgecolor='white')
    axes[i].hist(dead[col].dropna(), bins=30, alpha=0.6, color='#E45756',
                 label='Dead', density=True, edgecolor='white')
    axes[i].set_title(col.replace('_first', '').replace('_', ' ').title())
    axes[i].set_ylabel('Density')
    if i == 0: axes[i].legend()
for j in range(i + 1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Laboratory Values (First in H0-H24) by Outcome', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# GCS score distribution
gcs_cols = [c for c in df.columns if c.startswith('gcs_')]
if gcs_cols:
    fig, axes = plt.subplots(1, len(gcs_cols), figsize=(5 * len(gcs_cols), 4))
    if len(gcs_cols) == 1: axes = [axes]
    for i, col in enumerate(gcs_cols):
        label = col.replace('_min', ' (worst)').upper()
        for outcome, color, lbl in [(0, '#4C78A8', 'Alive'),
                                    (1, '#E45756', 'Dead')]:
            subset = df[df['in_hospital_death'] == outcome][col].dropna()
            vals, counts = np.unique(subset, return_counts=True)
            total = counts.sum()
            offset = -0.15 if outcome == 0 else 0.15
            axes[i].bar(vals + offset, counts / total, width=0.3,
                        color=color, alpha=0.7, label=lbl)
        axes[i].set_title(label)
        axes[i].set_xlabel('Score')
        axes[i].set_ylabel('Proportion')
        axes[i].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        if i == 0: axes[i].legend()
    plt.suptitle('Glasgow Coma Scale (Worst in H0-H24)', y=1.02)
    plt.tight_layout(); plt.show()

---
## 6. Missing Data Analysis

In [ ]:
analysis_cols = [c for c in df.columns
    if c not in ['visit_occurrence_id', 'person_id', 'sex',
                 'in_hospital_death', 'visit_start_datetime',
                 'admit_date', 'age_group']]

missing = df[analysis_cols].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#E45756' if v >= 30 else '#F58518' if v >= 10 else '#4C78A8'
          for v in missing.values]
ax.barh(range(len(missing)), missing.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(missing)))
ax.set_yticklabels(missing.index, fontsize=8)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Data by Feature')
ax.axvline(x=30, color='red', linestyle='--', alpha=0.5, label='30% threshold')
ax.legend(); ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# Is missingness correlated with outcome?
missing_by_outcome = df.groupby('in_hospital_death')[analysis_cols].apply(
    lambda x: x.isnull().mean() * 100
).T
missing_by_outcome.columns = ['Alive (%)', 'Dead (%)']
missing_by_outcome['Difference'] = (
    missing_by_outcome['Dead (%)'] - missing_by_outcome['Alive (%)']
).round(1)
missing_by_outcome = missing_by_outcome.sort_values(
    'Difference', ascending=False)
print("Missing data by outcome (top differences):")
print(missing_by_outcome.head(10).round(1).to_string())

---
## 7. Correlation Analysis

In [ ]:
numeric_cols = [c for c in analysis_cols
    if df[c].dtype in ['float64', 'int64', 'float32']]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=90, fontsize=7)
ax.set_yticklabels(numeric_cols, fontsize=7)
ax.set_title('Feature Correlation Matrix')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# Highly correlated pairs (|r| > 0.7)
high_corr = []
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append({'Feature 1': numeric_cols[i],
                'Feature 2': numeric_cols[j], 'Correlation': round(r, 3)})

if high_corr:
    high_corr_df = pd.DataFrame(high_corr).sort_values(
        'Correlation', key=abs, ascending=False)
    print(f"Highly correlated pairs (|r| > 0.7): {len(high_corr_df)}")
    print(high_corr_df.to_string(index=False))
else:
    print("No highly correlated pairs found (|r| > 0.7).")

---
## 8. Outcome Analysis

In [ ]:
# Table 1: descriptive statistics by outcome
table1_rows = []
for col in numeric_cols:
    a = alive[col].dropna()
    d = dead[col].dropna()
    table1_rows.append({
        'Variable': col,
        'Alive (mean +/- SD)': f"{a.mean():.1f} +/- {a.std():.1f}",
        'Dead (mean +/- SD)': f"{d.mean():.1f} +/- {d.std():.1f}",
        'Alive median [IQR]': f"{a.median():.1f} [{a.quantile(0.25):.1f}-{a.quantile(0.75):.1f}]",
        'Dead median [IQR]': f"{d.median():.1f} [{d.quantile(0.25):.1f}-{d.quantile(0.75):.1f}]",
    })

table1 = pd.DataFrame(table1_rows)
print("Table 1: Patient Characteristics by Outcome")
print("=" * 100)
print(table1.to_string(index=False))

In [ ]:
# Box plots: key features by outcome
key_features = [c for c in ['age', 'hr_mean', 'sbp_mean', 'resp_rate_mean',
    'creatinine_first', 'bun_first', 'wbc_first', 'potassium_first']
    if c in df.columns]

n = len(key_features); ncols = 4; nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()
for i, col in enumerate(key_features):
    bp = axes[i].boxplot([alive[col].dropna(), dead[col].dropna()],
        labels=['Alive', 'Dead'], patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#4C78A8')
    bp['boxes'][1].set_facecolor('#E45756')
    for b in bp['boxes']: b.set_alpha(0.7)
    axes[i].set_title(col.replace('_mean', '').replace('_first', '')
        .replace('_', ' ').title())
for j in range(i + 1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Feature Distributions by Outcome', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Univariate association with mortality (point-biserial correlation)
from scipy import stats

outcome_corr = []
for col in numeric_cols:
    valid = df[[col, 'in_hospital_death']].dropna()
    if len(valid) > 10:
        r, p = stats.pointbiserialr(valid['in_hospital_death'], valid[col])
        outcome_corr.append({'Feature': col, 'r': round(r, 3),
            'p-value': round(p, 4)})

outcome_corr_df = pd.DataFrame(outcome_corr).sort_values(
    'r', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E45756' if r > 0 else '#4C78A8'
          for r in outcome_corr_df['r']]
ax.barh(range(len(outcome_corr_df)), outcome_corr_df['r'].values,
        color=colors, edgecolor='white')
ax.set_yticks(range(len(outcome_corr_df)))
ax.set_yticklabels(outcome_corr_df['Feature'].values, fontsize=8)
ax.set_xlabel('Point-biserial Correlation')
ax.set_title('Feature Correlation with In-Hospital Mortality')
ax.axvline(x=0, color='black', linewidth=0.5); ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("\nCorrelation with mortality:")
print(outcome_corr_df.to_string(index=False))

---
## 9. Outlier Detection

In [ ]:
PLAUSIBLE_RANGES = {
    'hr_mean': (20, 300), 'sbp_mean': (40, 300), 'dbp_mean': (10, 200),
    'temp_mean': (30, 45), 'resp_rate_mean': (4, 60), 'spo2_mean': (50, 100),
    'sodium_first': (100, 180), 'potassium_first': (1.5, 10),
    'glucose_first': (20, 1000), 'creatinine_first': (0.1, 30),
}

print("Outlier check (values outside plausible clinical range):")
print(f"{'Feature':<25} {'Range':>15} {'N outliers':>12} {'%':>8}")
print("-" * 62)
for col, (lo, hi) in PLAUSIBLE_RANGES.items():
    if col in df.columns:
        vals = df[col].dropna()
        n_out = ((vals < lo) | (vals > hi)).sum()
        pct = 100 * n_out / len(vals) if len(vals) > 0 else 0
        print(f"{col:<25} {'[' + str(lo) + ', ' + str(hi) + ']':>15} "
              f"{n_out:>12} {pct:>7.1f}%")

---
## Summary

**Data cleaning strategy for modeling:**

- **Missing values**: Features with > 30% missing excluded. Remaining NAs imputed with median.
- **Multicollinearity**: Pairs with |r| > 0.85 flagged for review.
- **Outliers**: Clinical plausibility checked. No automatic removal.
- **Class imbalance**: Low mortality rate noted. Stratified train/test split required.

**Next step:** Open `02_ml_mortality.qmd` for the machine learning pipeline.

In [ ]:
usable = [c for c in numeric_cols if df[c].isnull().mean() < 0.30]
print("=" * 60)
print("EDA COMPLETE")
print("=" * 60)
print(f"\nDataset: {df.shape[0]} visits, {len(numeric_cols)} numeric features")
print(f"Mortality: {int(df['in_hospital_death'].sum())} / {len(df)} "
      f"({100 * df['in_hospital_death'].mean():.1f}%)")
print(f"Usable features (<30% missing): {len(usable)}")
print(f"CSV exported to: data/datasets/mortality_dataset.csv")